# Archive - not used by the current analysis
Kept in case something is wanted back. Mostly relies on variables (`flow_models`, `gauge_loaders`, `test_dataloader`, `Caravan_Attributes`) that no longer exist, so these will not run as-is.

### original cell 48

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

def plot_cdfs_individual(
    avg_metrics_by_results,
    metrics=["KGE"], #"NSE", 
    figsize_per_plot= (12, 6),
    title_names=None,
    model_names=None,
    model_colors=None,
):
    """
    Plot and SAVE individual CDFs of specified metrics for each results dictionary.
    """
    # ── font sizes ────────────────────────────────────────────────────────────
    fs_title  = 20
    fs_xlabel = 18
    fs_ylabel = fs_xlabel
    fs_tick   = 16
    fs_legend = 18
    # ─────────────────────────────────────────────────────────────────────────

    if model_colors is None:
        # Assuming MODEL_COLORS is defined globally in your script
        model_colors = MODEL_COLORS 

    results_names = list(avg_metrics_by_results.keys())
    n_rows = len(results_names)

    if title_names is None:
        title_names = results_names
    if len(title_names) != n_rows:
        raise ValueError(
            f"title_names has {len(title_names)} entries but there are {n_rows} results dicts."
        )

    # fallback colour cycle for any model not in model_colors
    fallback_colors = cm.tab10(np.linspace(0, 0.9, 10))
    
    for results_name, display_title in zip(results_names, title_names):

        model_dict = avg_metrics_by_results[results_name]
        models = list(model_dict.keys())

        if model_names is None:
            display_model_names = models
        else:
            if len(model_names) != len(models):
                raise ValueError(
                    f"model_names has {len(model_names)} entries but '{results_name}' "
                    f"has {len(models)} models."
                )
            display_model_names = model_names

        for metric in metrics:
            
            # 1. Create a NEW figure for this specific metric and dataset
            fig, ax = plt.subplots(figsize=figsize_per_plot)
            fallback_idx = 0 

            for model, display_name in zip(models, display_model_names):
                df = model_dict[model]

                if metric not in df.columns:
                    print(
                        f"Warning: '{metric}' not found for model '{model}' "
                        f"in '{results_name}'. Skipping."
                    )
                    continue

                # resolve colour: use model_colors dict, fall back to tab10
                if display_name in model_colors:
                    color = model_colors[display_name]
                else:
                    color = fallback_colors[fallback_idx % 10]
                    fallback_idx += 1

                values = df[metric].dropna().values
                
                # Skip plotting if the data is entirely empty
                if len(values) == 0:
                    continue
                    
                sorted_vals = np.sort(values)
                cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)

                ax.plot(sorted_vals, cdf, label=display_name, color=color, linewidth=1.8)

            ax.set_xlabel(metric, fontsize=fs_xlabel)
            ax.set_ylabel("CDF", fontsize=fs_ylabel)
            ax.set_title(f"{display_title}  —  {metric}", fontsize=fs_title, fontweight="bold")
            ax.tick_params(labelsize=fs_tick)
            ax.set_ylim(0, 1)
            ax.grid(True, linestyle="--", alpha=0.4)
            ax.spines[["top", "right"]].set_visible(False)

            # 1. Extract the handles and labels
            handles, labels = ax.get_legend_handles_labels()
            
            # Only draw legends if there is actually data on the plot
            if handles:
                # 2. Create the top legend (First 3 models)
                leg1 = ax.legend(
                    handles[:3], labels[:3],
                    fontsize=fs_legend, 
                    loc="upper center",          
                    bbox_to_anchor=(0.5, -0.13), # Sit right below the plot
                    ncol=3,                      
                    framealpha=0.7
                )
                
                # 3. Add the top legend manually
                ax.add_artist(leg1)
                
                # 4. Create the bottom legend (All remaining models)
                if len(handles) > 3:
                    remaining_count = len(handles[3:])
                    leg2 = ax.legend(
                        handles[3:], labels[3:],
                        fontsize=fs_legend, 
                        loc="upper center",          
                        bbox_to_anchor=(0.5, -0.25), # Push this one down below the first legend
                        ncol=remaining_count,        
                        framealpha=0.7
                    )
            
            # Clean up the layout
            fig.tight_layout()
            
            # 5. Save the individual plot!
            # Cleans up the filename to avoid weird spaces or characters
            safe_results_name = results_name.replace(" ", "_").replace("/", "_")
            filename = f"CDF_{safe_results_name}_{metric}.png"
            
            # bbox_inches='tight' is CRITICAL here so your bottom legend isn't cut off!
            fig.savefig(filename, dpi=300, bbox_inches='tight')
            
            # 6. Close the figure to free memory before the loop starts again
            plt.close(fig)
            
            print(f"Saved: {filename}")



plot_cdfs_individual(
        avg_metrics_by_results,
        title_names=[
            "No flow – Reservoir",
            # "Flow – Reservoir",
            "No flow – Non-Reservoir",
            # "Flow – Non-Reservoir",
        ],
        model_names=model_names,
        model_colors=MODEL_COLORS,
    )

### original cell 11

In [ ]:
# IDs, val as scotland
test_res_tensor_metadata = torch.load(
    "/perm/mokr/CamelsGB_V2/test_res_tensor_metadata.pt"
)
test_non_res_tensor_metadata = torch.load(
    "/perm/mokr/CamelsGB_V2/test_non_res_tensor_metadata.pt"
)


# 1. Pre-calculate the mapping once
gauge_index_map = get_consecutive_gauge_indices(test_res_tensor_metadata)

gauge_loaders = {}
for gid, indices in gauge_index_map.items():
    # Use Subset to point to the specific data for this gauge
    gauge_subset = Subset(test_non_res_dataset, indices)

    # Create the loader (Batch size = total gauge samples for single-pass evaluation)
    gauge_loaders[gid] = DataLoader(
        gauge_subset, batch_size=len(indices), shuffle=False
    )

from collections import defaultdict

### original cell 86

In [ ]:
# Font sizes for paper-quality plots
TITLE_FONTSIZE = 20
LABEL_FONTSIZE = 18
TICK_FONTSIZE = 16
LEGEND_FONTSIZE = 14

# Color scheme
ECMWF_COLORS = ["#2E86AB", "#F18F01", "#06A77D", "#A23B72", "#FFD700"]
TARGET_COLOR = "black"

# Model name mappings (customize as needed)


def plot_model_comparison_timeseries(
    batch_outputs,
    model_names,
    gauge_name,
    start=0,
    end=365,
    model_name_map=None,
    colors=None,
    figsize=(15, 8),
    save_path=None,
    dpi=300,
):
    """
    Plot time series comparison of multiple models against target with paper-ready formatting.

    Parameters:
    -----------
    batch_outputs : dict
        Dictionary with model names as keys, containing (output, target) tuples
    model_names : list
        List of model names to plot
    gauge_name : str
        Name of the gauge/catchment for the title
    start : int
        Starting time step (default: 0)
    end : int
        Ending time step (default: 365)
    model_name_map : dict, optional
        Dictionary to rename models in legend. If None, uses MODEL_NAME_MAP
    colors : list, optional
        List of colors for models. If None, uses ECMWF_COLORS
    figsize : tuple
        Figure size (width, height) in inches
    save_path : str, optional
        Path to save figure. If None, displays instead
    dpi : int
        Resolution for saved figure (default: 300)
    """
    if model_name_map is None:
        model_name_map = MODEL_NAME_MAP

    if colors is None:
        colors = ECMWF_COLORS

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)

    # Get target - take the last timestep for each sequence
    target = batch_outputs[model_names[0]][0][1][start:end, -1].cpu().numpy()

    # Plot target in black with thicker line
    ax.plot(target, label="Observed", linewidth=3, color=TARGET_COLOR, zorder=10)

    # Plot each model's output
    for idx, model_name in enumerate(model_names):
        output = batch_outputs[model_name][0][0][start:end, -1].cpu().numpy()

        # Customize line style based on model name
        if "Full" in model_name or "Hydra" in model_name:
            linestyle = "--"
        else:
            linestyle = "-"

        # Use color from palette
        color = colors[idx % len(colors)]

        # Use mapped name if available
        display_name = model_name_map.get(model_name, model_name)

        ax.plot(
            output,
            label=display_name,
            linewidth=2.5,
            linestyle=linestyle,
            color=color,
            alpha=0.85,
        )

    # Labels and title with larger fonts
    ax.set_title(
        f"Model Predictions for {gauge_name}",
        fontsize=TITLE_FONTSIZE,
        fontweight="bold",
        pad=15,
    )
    ax.set_xlabel("Time Step (days)", fontsize=LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Streamflow (mm/day)", fontsize=LABEL_FONTSIZE, fontweight="bold")

    # Larger tick labels
    ax.tick_params(axis="both", which="major", labelsize=TICK_FONTSIZE)

    # Larger legend with better positioning
    ax.legend(
        fontsize=LEGEND_FONTSIZE,
        frameon=True,
        shadow=True,
        loc="best",
        framealpha=0.9,
        ncol=1,
    )

    # Grid with better visibility
    ax.grid(True, alpha=0.4, linestyle="--", linewidth=0.8)

    plt.tight_layout()

    # Save or show
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {save_path}")
    else:
        plt.show()

    return fig

### original cell 88

In [ ]:
gauges = np.unique(test_res_tensor_metadata["gauge_id"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_plots = 12

WARMUP = 10
MAX_BATCHES = 1

for i in range(num_plots):
    gauge = np.random.choice(gauges)
    gauge_name = get_basin_name(int(gauge))
    dataloader = gauge_loaders[gauge]

    batch_outputs = process_validation_batches(
        dataloader,
        model_names,
        flow_models,
        WARMUP,
        scaler_river,
        device=DEVICE,
        time_interval=1,
        max_batches=MAX_BATCHES,
        use_flow=True,
    )

    structured_batch_outputs = collect_validation_results(batch_outputs)
    
    fig = plot_model_comparison_timeseries(
        structured_batch_outputs, model_names, gauge_name, start=0, end=195
    )

### original cell 91

In [ ]:
def collect_validation_results(validation_generator):
    """
    Consumes the validation generator and formats the outputs into a nested 
    dictionary for the plotting function.
    
    Returns:
    --------
    dict
        Format: {model_name: [(output_batch_0, target_batch_0), (output_batch_1, target_batch_1), ...]}
    """
    batch_outputs = {}
    
    # Iterate through the yielded batches from your validation function
    for batch_results, target in validation_generator:
        
        for model_name, output in batch_results.items():
            # Initialize the list for the model if it doesn't exist yet
            if model_name not in batch_outputs:
                batch_outputs[model_name] = []
                
            # Append the (output, target) tuple for this specific batch
            batch_outputs[model_name].append((output, target))
            
    return batch_outputs

# 1. Initialize your generator
# (Make sure to pass all your specific arguments like models, dataloader, etc.)
validation_gen = process_validation_batches(
    dataloader,
    model_names,
    flow_models,
    WARMUP,
    scaler_river,
    device=DEVICE,
    time_interval=1,
    max_batches=MAX_BATCHES,
    use_flow=True,
)


# 2. Collect and structure the data using the wrapper
print("Collecting batch outputs...")
structured_batch_outputs = collect_validation_results(validation_gen)

# 3. Plot the results using the corrected plotting function
# (This will plot the first sequence in the first batch)
print("Generating plot...")
fig = plot_model_comparison_timeseries(
    structured_batch_outputs, model_names, gauge_name, start=0, end=95
)

### original cell 94

In [ ]:
def calculate_kge(predictions, targets):

    eps = 1e-6
    """Calculate KGE components (alpha, beta, and correlation)"""
    # Correlation coefficient
    pred_mean = np.mean(predictions)
    target_mean = np.mean(targets)

    numerator = np.mean((predictions - pred_mean) * (targets - target_mean))
    pred_std = np.sqrt(np.mean((predictions - pred_mean) ** 2))
    target_std = np.sqrt(np.mean((targets - target_mean) ** 2))
    correlation = numerator / ((pred_std * target_std) + eps)

    # Alpha (variability ratio)
    gamma = (pred_std / (pred_mean + eps)) / (target_std / (target_mean + eps))

    # Beta (bias ratio)
    beta = pred_mean / (target_mean + eps)

    kge = 1 - np.sqrt((correlation - 1) ** 2 + (gamma - 1) ** 2 + (beta - 1) ** 2)
    # print(gamma, beta, correlation)
    return kge


def calculate_nse(predictions, targets):
    eps = 1e-8
    """Calculate Nash-Sutcliffe Efficiency"""
    numerator = np.mean((predictions - targets) ** 2)
    denominator = np.mean((targets - np.mean(targets)) ** 2)
    return 1 - ((numerator + eps) / (denominator + eps))

### original cell 95

In [ ]:
import warnings

max_batches = 15
seed = 44
warmup = 0
loss_interval = 2
# This will suppress all warnings that occur while the function is running
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    # Reservoir model
    reservoir_outputs, targets, gauges, countries = validate_model(
        Reservoir_Embedded_LSTM,
        test_dataloader,
        criterion,
        max_batches,
        warmup,
        10,
        time_interval=15,
        share_location=True,
    )
    unnormalized_reservoir_outputs = streamflow_scaler.inverse_transform(
        reservoir_outputs.detach().numpy()
    )
    unnormalized_reservoir_outputs = np.clip(
        unnormalized_reservoir_outputs, a_min=0, a_max=None
    )  # <-- Clip below 0
    unnormalized_targets = streamflow_scaler.inverse_transform(targets.detach().numpy())

    # Non-reservoir model
    nonreservoir_outputs, targets = validate_model(
        NonReservoir_Embedded_LSTM,
        test_dataloader,
        criterion,
        max_batches,
        warmup,
        loss_interval,
        time_interval=15,
    )
    unnormalized_nonreservoir_outputs = streamflow_scaler.inverse_transform(
        nonreservoir_outputs.detach().numpy()
    )
    unnormalized_nonreservoir_outputs = np.clip(
        unnormalized_nonreservoir_outputs, a_min=0, a_max=None
    )  # <-- Clip

    # Full model
    full_outputs, targets = validate_model(
        Full_Embedded_LSTM,
        test_dataloader,
        criterion,
        max_batches,
        warmup,
        loss_interval,
        time_interval=15,
    )
    unnormalized_full_outputs = streamflow_scaler.inverse_transform(
        full_outputs.detach().numpy()
    )
    unnormalized_full_outputs = np.clip(
        unnormalized_full_outputs, a_min=0, a_max=None
    )  # <-- Clip

    # Hydra model
    hydra_res_outputs, targets = validate_model(
        [Hydra_Body, Hydra_ResHead],
        test_dataloader,
        criterion,
        max_batches,
        warmup,
        loss_interval,
        time_interval=15,
    )
    unnormalized_hydra_res_outputs = streamflow_scaler.inverse_transform(
        hydra_res_outputs.detach().numpy()
    )
    unnormalized_hydra_res_outputs = np.clip(
        unnormalized_hydra_res_outputs, a_min=0, a_max=None
    )  # <-- Clip

    # set_seeds(seed=42)
    # hydra_res_outputs, targets = validate_model(Hydra_Main, test_dataloader, criterion, max_batches, warmup, loss_interval, time_interval = 15)
    # unnormalized_hydra_res_outputs = streamflow_scaler.inverse_transform(hydra_res_outputs.detach().numpy())


import matplotlib.pyplot as plt

for i in range(1, max_batches):

    plt.figure(figsize=(12, 6))

    kge_reservoir = calculate_kge(
        unnormalized_reservoir_outputs[i], unnormalized_targets[i]
    )
    kge_nonreservoir = calculate_kge(
        unnormalized_nonreservoir_outputs[i], unnormalized_targets[i]
    )
    kge_full = calculate_kge(unnormalized_full_outputs[i], unnormalized_targets[i])
    kge_hydra = calculate_kge(
        unnormalized_hydra_res_outputs[i], unnormalized_targets[i]
    )

    nse_reservoir = calculate_nse(
        unnormalized_reservoir_outputs[i], unnormalized_targets[i]
    )
    nse_nonreservoir = calculate_nse(
        unnormalized_nonreservoir_outputs[i], unnormalized_targets[i]
    )
    nse_full = calculate_nse(unnormalized_full_outputs[i], unnormalized_targets[i])
    nse_hydra = calculate_nse(
        unnormalized_hydra_res_outputs[i], unnormalized_targets[i]
    )

    model_names = [
        "Reservoir_Embedded_LSTM",
        "NonReservoir_Embedded_LSTM",
        "Full_Embedded_LSTM",
        "Hydra_ResHead",
        "Hydra_Main",
    ]
    colors = ["orange", "green", "red", "purple", "blue"]

    # Plot targets and outputs with metrics in legend labels
    plt.plot(unnormalized_targets[i], label="Target", color="black", linewidth=2)
    plt.plot(
        unnormalized_reservoir_outputs[i],
        label=f"Reservoir-LSTM (NSE: {nse_reservoir:.3f}, KGE: {kge_reservoir:.3f})",
        color="orange",
        linestyle="--",
        linewidth=2,
    )
    plt.plot(
        unnormalized_nonreservoir_outputs[i],
        label=f"NonReservoir-LSTM (NSE: {nse_nonreservoir:.3f}, KGE: {kge_nonreservoir:.3f})",
        color="green",
        linestyle="--",
        linewidth=2,
    )
    plt.plot(
        unnormalized_full_outputs[i],
        label=f"Full-LSTM (NSE: {nse_full:.3f}, KGE: {kge_full:.3f})",
        color="red",
        linewidth=2,
        linestyle="--",
    )
    plt.plot(
        unnormalized_hydra_res_outputs[i],
        label=f"Hydra (NSE: {nse_hydra:.3f}, KGE: {kge_hydra:.3f})",
        color="purple",
        linewidth=2,
        linestyle="--",
    )

    # Add labels and title
    plt.title(f"Streamflow for Basin: {gauges[i]}")
    plt.xlabel("Date")
    plt.ylabel("Specific Discharge ($ms^{-1}$)")
    plt.legend()
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)

    plt.tight_layout()
    plt.show()

### original cell 96

In [ ]:
# samples = sample_basin_date_pairs('NonReservoirs_Train', n_samples=10, num_basins=90)

for basin in [
    "hysets_11443000"
]:  # basins hysets_12457000 1976 hysets_12199000 hysets_11443000 camelsch_3019
    region = basin.split("_")[0]

    # Static data

    basin_caravan_attributes = Caravan_Attributes[region][
        Caravan_Attributes[region]["gauge_id"] == basin
    ].copy()
    basin_hydroatlas_attributes = Hydroatlas_Attributes[region][
        Hydroatlas_Attributes[region]["gauge_id"] == basin
    ].copy()
    basin_other_attributes = Other_Attributes[region][
        Other_Attributes[region]["gauge_id"] == basin
    ].copy()

    static_df = pd.concat(
        [basin_caravan_attributes, basin_hydroatlas_attributes, basin_other_attributes],
        axis=1,
    )
    static_df = static_df.select_dtypes(include=["number"])
    static_df = static_df.loc[:, static_df.columns.isin(ordered_static_cols)]
    static_df = static_df.reindex(
        columns=ordered_static_cols
    )  # Ensure consistent column order
    static_input = torch.tensor(
        static_df.values, dtype=torch.float32
    )  # shape: (1, num_features)

    # Temporal data
    start = pd.to_datetime("1956-01-01")
    end = start + pd.Timedelta(days=365)
    basin_temporal = group_datasets[basin].sel(date=slice(start, end))

    plot_dates = basin_temporal.date.values

    if "potential_evaporation_sum_ERA5_LAND" in basin_temporal:
        basin_temporal = basin_temporal.rename(
            {"potential_evaporation_sum_ERA5_LAND": "potential_evaporation_sum"}
        )

    basin_temporal_input = basin_temporal[ordered_temporal_vars]
    targets = basin_temporal["streamflow"]
    # temporal_input = basin_temporal_input.drop_vars('streamflow')

    # Prepare tensors for the model
    temporal_np = basin_temporal_input.to_array().transpose("date", "variable").values
    temporal_input = torch.tensor(temporal_np, dtype=torch.float32).unsqueeze(0)
    target = torch.tensor(targets.values, dtype=torch.float32).unsqueeze(0)

    temporal_scaled, static_scaled, targets_scaled, _ = scale_data(
        temporal_tensor=temporal_input,
        static_tensor=static_input,
        targets_tensor=target,
        scaler_dir=SCALER_DIRECTORY,
        device="cpu",
    )

    models = [
        Reservoir_Embedded_LSTM,
        NonReservoir_Embedded_LSTM,
        Full_Embedded_LSTM,
        [Hydra_Body, Hydra_ResHead],
        Hydra_Main,
    ]
    model_names = [
        "Reservoir_Embedded_LSTM",
        "NonReservoir_Embedded_LSTM",
        "Full_Embedded_LSTM",
        "Hydra_ResHead",
        "Hydra_Main",
    ]
    colors = ["orange", "green", "red", "purple", "blue"]

    models = [NonReservoir_Embedded_LSTM, Full_Embedded_LSTM, Hydra_Main]
    model_names = ["NonReservoir_Embedded_LSTM", "Full_Embedded_LSTM", "Hydra_Main"]
    colors = ["green", "red", "blue"]

    plt.figure(figsize=(12, 6))

    # Plot each model
    for i, (model, name, color) in enumerate(zip(models, model_names, colors)):
        # Run the model pipeline
        if isinstance(model, list):  # Handle the Hydra case
            model_outputs = run_model_pipeline(
                model, static_scaled, temporal_scaled
            ).squeeze(-1)
        else:
            model_outputs = run_model_pipeline(
                model, static_scaled, temporal_scaled
            ).squeeze(-1)

        # Unnormalize outputs
        unnormalized_outputs = streamflow_scaler.inverse_transform(
            model_outputs.detach().numpy()
        )
        unnormalized_outputs = np.clip(unnormalized_outputs, a_min=0, a_max=None)

        # Plot the line
        kge = calculate_nse(unnormalized_outputs, targets.values)
        calculate_kge(unnormalized_outputs, targets.values)
        # Plot the line with KGE in the label
        plt.plot(
            plot_dates,
            unnormalized_outputs.squeeze(),
            linestyle="--",
            label=f"{name} (NSE: {kge:.3f})",
            color=color,
        )

    # Plot observed streamflow
    plt.plot(
        plot_dates,
        targets.values,
        label="Observed Streamflow",
        color="black",
        linewidth=2,
    )

    plt.xlabel("Date")
    plt.ylabel("Specific Discharge ($ms^{-1}$)")
    plt.legend()

    plt.title(
        f"Predicted vs Target Streamflow for Basin: {basin}, Start Date: {start.strftime('%Y-%m-%d')}"
    )

    # A more robust title if you comment out predictions:
    plt.title(
        f"Observed Streamflow for Basin: {basin}\nStart Date: {start.strftime('%Y-%m-%d')}"
    )

    plt.grid(
        True, which="both", linestyle="--", linewidth=0.5
    )  # Add grid for better readability
    plt.tight_layout()
    plt.show()